# linalg-solve-batched — ex2: batched solve with multiple right-hand sides

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `linalg-solve-batched`. Running the final beacon cell reports progress against the `PyTorch: Batched linalg.solve` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Batched linalg.solve` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`linalg-solve-batched`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "linalg-solve-batched"
DD_SUBTOPIC = "PyTorch: Batched linalg.solve"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `t.linalg.solve` — quick refresher

`t.linalg.solve(A, b)` solves `A @ x == b` for `x`. Shape contract:
- `A: (..., n, n)` — leading dims are the batch.
- `b: (..., n)`    → returns `x: (..., n)`         (one RHS per batch)
- `b: (..., n, m)` → returns `x: (..., n, m)`      (m RHS columns per batch)

**Multiple right-hand sides.** When `b` carries a trailing `m` axis, the LU factorization of each `A[..., :, :]` is reused for all `m` substitutions. This is how you compute a matrix inverse efficiently (`solve(A, I)`) — and it's exactly the shape ARENA's triangle-mesh rasterization needs for batched barycentric solves with multiple query points per triangle.

### Exercise 2 — batched solve with multiple right-hand sides

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `t.linalg.solve` with `A: (K, n, n)` and `b: (K, n, m)` to solve `K * m` linear systems sharing `K` LU factorizations, returning `x: (K, n, m)`.
> Keywords: linalg, solve, multi-rhs, inverse
> ```

**KCs targeted:** `linalg-solve-leading-batch`, `linalg-solve-shape-contract`

Implement `ex2_solve_multi_rhs(A, B)`.

- `A` has shape `(K, n, n)` — `K` square matrices.
- `B` has shape `(K, n, m)` — `m` right-hand sides per matrix.
- Return shape `(K, n, m)`: column `j` of slice `k` is the solution to `A[k] @ x == B[k, :, j]`.

This is the shape contract `t.linalg.solve` already supports — pass `A` and `B` as-is, no per-column loop.

**Why this is fast.** Each `A[k]` is LU-factorized ONCE; the `m` triangular solves share that factorization. If you looped over `j` you'd pay `m` extra LU factorizations per batch, trashing performance for `m >= 4`.

Assume all `A[k]` are non-singular.

In [ ]:
def ex2_solve_multi_rhs(A: Tensor, B: Tensor) -> Tensor:
    """Solve K batched n×n systems with m RHS columns each."""
    raise NotImplementedError()


def _test_ex2():
    # Hand-checked tiny case: K=2 systems, n=2, m=3 RHSs.
    A = t.tensor([
        [[2.0, 0.0], [0.0, 3.0]],     # diagonal — easy to verify
        [[1.0, 1.0], [0.0, 2.0]],
    ])
    B = t.tensor([
        [[2.0, 4.0, 6.0], [3.0, 6.0, 9.0]],   # A[0] solution = [1,2,3] / [1,2,3]
        [[1.0, 0.0, 5.0], [2.0, 4.0, 2.0]],
    ])
    X = ex2_solve_multi_rhs(A, B)
    assert tuple(X.shape) == (2, 2, 3), f'expected (2,2,3), got {tuple(X.shape)}'
    # Verify A @ X == B for every batch and column.
    AX = A @ X
    assert t.allclose(AX, B, atol=1e-5), (
        f'A @ X != B:\nAX = {AX}\nB = {B}'
    )

    # Special case — solve A x = I gives A^{-1}.
    rng = t.Generator().manual_seed(7)
    K, n = 4, 3
    Abig = t.randn(K, n, n, generator=rng)
    # Nudge away from singular by adding scaled identity.
    Abig = Abig + 2.0 * t.eye(n).expand(K, n, n)
    I = t.eye(n).expand(K, n, n).contiguous()
    Ainv = ex2_solve_multi_rhs(Abig, I)
    assert tuple(Ainv.shape) == (K, n, n)
    should_be_I = Abig @ Ainv
    assert t.allclose(should_be_I, I, atol=1e-4), (
        f'A @ A^-1 != I:\n{should_be_I}'
    )

    # Vector-RHS (m=1) sanity: shape (K, n, 1) round-trips correctly.
    B1 = t.randn(K, n, 1, generator=rng)
    X1 = ex2_solve_multi_rhs(Abig, B1)
    assert tuple(X1.shape) == (K, n, 1)
    assert t.allclose(Abig @ X1, B1, atol=1e-4)
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_solve_multi_rhs(A: Tensor, B: Tensor) -> Tensor:
    return t.linalg.solve(A, B)
```

**One call — that's the whole answer.** The shape contract of `t.linalg.solve` already handles trailing `m` columns: when `B.shape == (..., n, m)`, the result is `(..., n, m)`.

**Why ARENA cares about this shape.** The triangle-rasterization drill solves a barycentric system per (triangle, pixel) pair. With `K` triangles and `m` query pixels, the natural batch is `A: (K, 2, 2)` and `B: (K, 2, m)`, returning `(K, 2, m)`. One `solve` call replaces a nested Python loop.

**Inverse via solve.** `A @ A^-1 = I` ⇒ `A^-1 = solve(A, I)`. This is the recommended way to compute inverses in PyTorch — explicit `t.linalg.inv` is implemented as `solve(A, I)` internally, and is more numerically stable than `1/A` or Gauss-Jordan elimination written by hand.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()